In [5]:
import os
import json
import re
from datetime import date

import pandas as pd
from openpyxl import Workbook
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter


# ============================================================
# Styling
# ============================================================
THIN = Side(style="thin", color="BFBFBF")
BORDER = Border(left=THIN, right=THIN, top=THIN, bottom=THIN)

HEADER_FILL = PatternFill("solid", fgColor="1F4E79")
HEADER_FONT = Font(bold=True, color="FFFFFF")
TITLE_FONT = Font(bold=True, size=14, color="1F4E79")
BOLD = Font(bold=True)
NOTE_FONT = Font(italic=True, color="666666")

CENTER = Alignment(horizontal="center", vertical="center", wrap_text=True)
LEFT = Alignment(horizontal="left", vertical="center", wrap_text=True)


def autosize_columns(ws, min_width=10, max_width=60):
    for col in ws.columns:
        col_letter = get_column_letter(col[0].column)
        max_len = 0
        for cell in col:
            v = "" if cell.value is None else str(cell.value)
            if len(v) > max_len:
                max_len = len(v)
        ws.column_dimensions[col_letter].width = max(min_width, min(max_width, max_len + 2))


def style_table(ws, start_row, start_col, end_row, end_col, header=True):
    for r in range(start_row, end_row + 1):
        for c in range(start_col, end_col + 1):
            cell = ws.cell(row=r, column=c)
            cell.border = BORDER
            if r == start_row and header:
                cell.fill = HEADER_FILL
                cell.font = HEADER_FONT
                cell.alignment = CENTER
            else:
                cell.alignment = LEFT


def write_df(ws, df, start_row=1, start_col=1, header=True):
    """Writes a pandas dataframe into a worksheet with basic table styling."""
    r = start_row
    for row in dataframe_to_rows(df, index=False, header=header):
        c = start_col
        for value in row:
            ws.cell(row=r, column=c, value=value)
            c += 1
        r += 1

    end_row = start_row + len(df) + (1 if header else 0) - 1
    end_col = start_col + df.shape[1] - 1

    if df.shape[1] > 0 and end_row >= start_row:
        style_table(ws, start_row, start_col, end_row, end_col, header=header)
    autosize_columns(ws)
    return end_row, end_col


# ============================================================
# Excel sheet name sanitization
# ============================================================
INVALID_SHEET_CHARS = r'[\[\]\:\*\?\/\\]'

def safe_sheet_name(name: str):
    """Excel: max 31 chars, no []:*?/\\ """
    name = "Sheet" if name is None else str(name)
    name = re.sub(INVALID_SHEET_CHARS, "-", name).strip()
    if len(name) > 31:
        name = name[:31]
    return name or "Sheet"


# ============================================================
# Data extractors (from v2 family block)
# ============================================================
def df_components(fam):
    rows = []
    comps = fam.get("components", {}) or {}
    for sole_type, comp_dict in comps.items():
        for comp_code, comp in (comp_dict or {}).items():
            rows.append({
                "SoleType": sole_type,
                "ComponentCode": comp_code,
                "ComponentName": comp.get("displayName"),
                "Compound": comp.get("compound"),
                "Notes": comp.get("notes")
            })
    return pd.DataFrame(rows)


def df_styles(fam):
    rows = []
    for s in fam.get("stylesUsingThisFamily", []) or []:
        rows.append({
            "StyleCode": s.get("styleCode"),
            "StyleName": s.get("styleName"),
            "Season": s.get("season"),
            "Notes": s.get("notes")
        })
    return pd.DataFrame(rows)


def df_sourcing(fam):
    """Support both legacy and current sourcingRules schemas."""
    rows = []
    for r in fam.get("sourcingRules", []) or []:
        rows.append({
            # New schema (from data_processing.ipynb)
            "FactoryNumber": r.get("factoryNumber"),
            "SoleType": r.get("soleType"),
            "SolePart": r.get("solePart"),
            "VendorId": r.get("vendorId"),
            # Legacy schema fallback
            "FactoryCode": r.get("factoryCode"),
            "ComponentCode": r.get("componentCode"),
            "VendorCode": r.get("vendorCode"),
            "EffectiveFrom": r.get("effectiveFrom"),
            "EffectiveTo": r.get("effectiveTo"),
            "Notes": r.get("notes")
        })
    return pd.DataFrame(rows)


def df_sizing_flat(fam):
    """
    Optional: Sizing Rule sheet as a flat table
    (ComponentCode, MoldSize) -> ShoeSizes (comma string), blank if null
    """
    rows = []
    sizing = fam.get("sizingRulesByComponent", {}) or {}
    for comp_code, block in sizing.items():
        sole_type = block.get("soleType")
        comp_name = block.get("componentName")
        notes = block.get("notes")
        ms_map = block.get("moldSizeToShoeSizes", {}) or {}
        for mold_size, shoe_sizes in ms_map.items():
            if shoe_sizes is None:
                shoe_str = ""
            else:
                shoe_str = ", ".join([str(x) for x in shoe_sizes])
            rows.append({
                "ComponentCode": comp_code,
                "SoleType": sole_type,
                "ComponentName": comp_name,
                "MoldSize": str(mold_size),
                "ShoeSizes": shoe_str,
                "Notes": notes
            })
    df = pd.DataFrame(rows)
    if not df.empty:
        def _to_float(x):
            try:
                return float(x)
            except:
                return 999999.0
        df["__m"] = df["MoldSize"].apply(_to_float)
        df = df.sort_values(["ComponentCode", "__m"]).drop(columns="__m")
    return df


def df_mold_record_details(fam):
    """
    Put detailed mold-record-level info here (useful in Basic_information).
    One row per mold record under each component.
    """
    rows = []
    comps = fam.get("components", {}) or {}
    for sole_type, comp_dict in comps.items():
        for comp_code, comp in (comp_dict or {}).items():
            for m in (comp.get("molds") or []):
                inv = m.get("inventory") or {}
                cap = m.get("capacity") or {}
                asset = m.get("asset") or {}
                loc = (m.get("location") or {}).get("name")

                rows.append({
                    "SoleType": sole_type,
                    "ComponentCode": comp_code,
                    "ComponentName": comp.get("displayName"),
                    "VendorName": m.get("vendorName") or m.get("vendorCode"),
                    "VendorCode": m.get("vendorCode"),
                    "MoldShop": m.get("moldShopName") or m.get("moldShopCode"),
                    "Location": loc,
                    "InitSeason": m.get("initSeason"),
                    "LastDemandSeason": m.get("lastDemandSeason"),
                    "TotalMoldQty": inv.get("totalMoldQty"),
                    "DailyOutput": cap.get("dailyOutput"),
                    "ActualOutput": cap.get("actualOutput"),
                    "MoldCost": asset.get("moldCost"),
                    "Ownership": asset.get("ownership"),
                    "Condition": asset.get("condition"),
                    "ConditionNote": asset.get("conditionNote"),
                    "Comments": m.get("comments"),
                    "Remark": m.get("remark")
                })
    return pd.DataFrame(rows)


# ============================================================
# Component inventory grid builder (VendorName-only, includes ShoeSizes)
# ============================================================
def build_component_grid_vendor_only(comp, sizing_block=None):
    """
    Output columns:
      MoldSize | ShoeSizes | <VendorName columns...>

    Vendor columns aggregate SUM across all mold records sharing same vendorName.
    ShoeSizes comes from sizing_block['moldSizeToShoeSizes'][moldSize]
      None -> blank
      list -> comma string
    """
    molds = comp.get("molds", []) or []

    # sizing map
    sizing_map = None
    if sizing_block:
        sizing_map = (sizing_block.get("moldSizeToShoeSizes") or {})

    # collect sizes from sizing keys + inventory keys
    all_sizes = set()
    if sizing_map is not None:
        all_sizes.update([str(k) for k in sizing_map.keys()])
    for m in molds:
        qty_map = ((m.get("inventory") or {}).get("qtyByMoldSize")) or {}
        all_sizes.update([str(k) for k in qty_map.keys()])

    def _to_float(s):
        try:
            return float(s)
        except:
            return 999999.0

    sizes_sorted = sorted(all_sizes, key=_to_float)

    # determine vendor columns (VendorName only)
    vendor_names = []
    for m in molds:
        vn = m.get("vendorName") or m.get("vendorCode") or "UNKNOWN_VENDOR"
        vendor_names.append(str(vn))
    vendor_cols = sorted(set(vendor_names))

    # aggregate qty by vendorName per moldSize
    agg = {v: {} for v in vendor_cols}
    for m in molds:
        vn = str(m.get("vendorName") or m.get("vendorCode") or "UNKNOWN_VENDOR")
        qty_map = ((m.get("inventory") or {}).get("qtyByMoldSize")) or {}

        for ms, qty in qty_map.items():
            ms = str(ms)
            if qty is None:
                continue
            # numeric sums if possible
            try:
                q = float(qty)
            except:
                q = qty  # non-numeric unusual

            if isinstance(q, (int, float)):
                agg[vn][ms] = agg[vn].get(ms, 0) + q
            else:
                agg[vn][ms] = q

    # build rows
    rows = []
    for ms in sizes_sorted:
        # shoe sizes string
        shoe_str = ""
        if sizing_map is not None and ms in sizing_map:
            v = sizing_map.get(ms)
            if v is None:
                shoe_str = ""
            else:
                shoe_str = ", ".join([str(x) for x in v])

        row = {"MoldSize": ms, "ShoeSizes": shoe_str}
        for vname in vendor_cols:
            row[vname] = agg.get(vname, {}).get(ms, None)
        rows.append(row)

    return pd.DataFrame(rows), vendor_cols


# ============================================================
# Export one family workbook
# ============================================================
def export_one_family(family_code, fam, output_dir, include_sizing_sheet=False):
    wb = Workbook()
    wb.remove(wb.active)

    # ----------------------------
    # Basic_information
    # ----------------------------
    ws_basic = wb.create_sheet("Basic_information")
    ws_basic["A1"] = f"Family: {family_code}"
    ws_basic["A1"].font = TITLE_FONT

    # header fields
    ws_basic["A3"] = "MoldCode"; ws_basic["B3"] = fam.get("moldCode")
    ws_basic["A4"] = "Brands"
    brands = fam.get("brands") or []
    ws_basic["B4"] = ", ".join([str(x) for x in brands]) if isinstance(brands, list) else str(brands)
    ws_basic["A5"] = "Notes"; ws_basic["B5"] = fam.get("notes")

    for r in range(3, 6):
        ws_basic[f"A{r}"].font = BOLD
        ws_basic[f"A{r}"].alignment = LEFT
        ws_basic[f"B{r}"].alignment = LEFT

    # Components table
    ws_basic["A7"] = "Components"
    ws_basic["A7"].font = TITLE_FONT
    df_comp = df_components(fam)
    if df_comp.empty:
        ws_basic["A8"] = "(no components found)"
    else:
        write_df(ws_basic, df_comp, start_row=8, start_col=1, header=True)
        ws_basic.freeze_panes = "A9"

    # Styles table
    df_sty = df_styles(fam)
    styles_start = (ws_basic.max_row or 1) + 3
    ws_basic[f"A{styles_start}"] = "Styles Using This Family"
    ws_basic[f"A{styles_start}"].font = TITLE_FONT
    if df_sty.empty:
        ws_basic[f"A{styles_start+1}"] = "(no styles listed)"
    else:
        write_df(ws_basic, df_sty, start_row=styles_start+1, start_col=1, header=True)

    # Mold record details table (reference)
    df_detail = df_mold_record_details(fam)
    detail_start = (ws_basic.max_row or 1) + 3
    ws_basic[f"A{detail_start}"] = "Mold Records Detail (Reference)"
    ws_basic[f"A{detail_start}"].font = TITLE_FONT
    ws_basic[f"A{detail_start+1}"] = "This section is for reference (mold shop, location, cost, condition...). Inventory editing is done in component sheets."
    ws_basic[f"A{detail_start+1}"].font = NOTE_FONT

    if df_detail.empty:
        ws_basic[f"A{detail_start+2}"] = "(no mold records)"
    else:
        write_df(ws_basic, df_detail, start_row=detail_start+2, start_col=1, header=True)

    autosize_columns(ws_basic)

    # ----------------------------
    # Sourcing Rule
    # ----------------------------
    ws_src = wb.create_sheet("Sourcing Rule")
    ws_src["A1"] = "Sourcing Rules (Factory × Component → Vendor)"
    ws_src["A1"].font = TITLE_FONT

    df_src = df_sourcing(fam)
    if df_src.empty:
        ws_src["A3"] = "(no sourcing rules yet)"
    else:
        write_df(ws_src, df_src, start_row=3, start_col=1, header=True)
        ws_src.freeze_panes = "A4"
    autosize_columns(ws_src)

    # ----------------------------
    # Optional Sizing Rule sheet (flat view)
    # (You asked to maintain sizing inside MoldInv; so default is False)
    # ----------------------------
    if include_sizing_sheet:
        ws_size = wb.create_sheet("Sizing Rule")
        ws_size["A1"] = "Sizing Rules (Optional view). Prefer editing ShoeSizes inside MoldInv_{component} sheets."
        ws_size["A1"].font = TITLE_FONT

        df_sz = df_sizing_flat(fam)
        if df_sz.empty:
            ws_size["A3"] = "(no sizing rules yet)"
        else:
            write_df(ws_size, df_sz, start_row=3, start_col=1, header=True)
            ws_size.freeze_panes = "A4"
        autosize_columns(ws_size)

    # ----------------------------
    # Mold Inventory sheets per component (combined sizing+inventory)
    # ----------------------------
    comps = fam.get("components", {}) or {}
    sizing_all = fam.get("sizingRulesByComponent", {}) or {}

    for sole_type, comp_dict in comps.items():
        for comp_code, comp in (comp_dict or {}).items():
            ws_inv = wb.create_sheet(safe_sheet_name(f"MoldInv_{comp_code}"))

            ws_inv["A1"] = f"Mold Inventory - {comp_code}"
            ws_inv["A1"].font = TITLE_FONT
            ws_inv["A2"] = f"SoleType: {sole_type}"; ws_inv["A2"].font = BOLD
            ws_inv["A3"] = f"Component Name: {comp.get('displayName')}"; ws_inv["A3"].font = BOLD
            ws_inv["A4"] = f"Compound: {comp.get('compound')}"; ws_inv["A4"].font = BOLD
            ws_inv["A5"] = "Edit ShoeSizes as comma-separated list (e.g., 3.5, 4). Leave blank if unknown. Vendor columns are aggregated by VendorName."
            ws_inv["A5"].font = NOTE_FONT

            # Get sizing block for this component (may not exist)
            sizing_block = sizing_all.get(comp_code)

            # Build combined grid (VendorName-only)
            df_grid, vendor_cols = build_component_grid_vendor_only(comp, sizing_block=sizing_block)

            if df_grid.empty:
                ws_inv["A7"] = "(no inventory or sizing keys found)"
                continue

            write_df(ws_inv, df_grid, start_row=7, start_col=1, header=True)
            ws_inv.freeze_panes = "C8"  # keeps MoldSize + ShoeSizes visible

            # Improve readability: set row height a bit
            ws_inv.row_dimensions[5].height = 30

            autosize_columns(ws_inv)

    # ----------------------------
    # Save workbook
    # ----------------------------
    os.makedirs(output_dir, exist_ok=True)
    out_path = os.path.join(output_dir, f"{family_code}.xlsx")
    wb.save(out_path)
    return out_path


def export_all_families(json_path, output_dir, include_sizing_sheet=False):
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    # support either root["families"] or root["data"]["families"] patterns
    families = data.get("families") or (data.get("data", {}) or {}).get("families")
    if not families:
        raise ValueError("No 'families' found in JSON.")

    written = []
    for family_code, fam in families.items():
        path = export_one_family(family_code, fam, output_dir, include_sizing_sheet=include_sizing_sheet)
        written.append(path)

    return written


# ============================================================
# Main
# ============================================================
# ===== EDIT THESE =====
JSON_PATH = "../data/mold_data.json"          # your master v2 JSON
OUTPUT_DIR = "../data/output/"     # output folder
INCLUDE_SIZING_SHEET = False         # True if you also want separate "Sizing Rule" sheet
# ======================

files = export_all_families(JSON_PATH, OUTPUT_DIR, include_sizing_sheet=INCLUDE_SIZING_SHEET)

print(f"Exported {len(files)} family workbook(s) to: {OUTPUT_DIR}")
for p in files:
    print(" -", p)

Exported 243 family workbook(s) to: ../data/output/
 - ../data/output/SA-1318.xlsx
 - ../data/output/SA-2017.xlsx
 - ../data/output/SA-2017-E.xlsx
 - ../data/output/SA-2253.xlsx
 - ../data/output/SA-2255.xlsx
 - ../data/output/SA-2255-4E.xlsx
 - ../data/output/SA-2301.xlsx
 - ../data/output/SA-2313.xlsx
 - ../data/output/SA-2408.xlsx
 - ../data/output/SA-2451.xlsx
 - ../data/output/SA-2511.xlsx
 - ../data/output/SA-2552.xlsx
 - ../data/output/SA-2556.xlsx
 - ../data/output/SA-2557.xlsx
 - ../data/output/SA-2558.xlsx
 - ../data/output/SA-2601.xlsx
 - ../data/output/SA-2603.xlsx
 - ../data/output/SA-2606.xlsx
 - ../data/output/SA-2651.xlsx
 - ../data/output/SA-2652.xlsx
 - ../data/output/SA-2653.xlsx
 - ../data/output/SA-2661.xlsx
 - ../data/output/SA-2204.xlsx
 - ../data/output/SA-2309.xlsx
 - ../data/output/SA-2409.xlsx
 - ../data/output/SA-2551.xlsx
 - ../data/output/SA-2608.xlsx
 - ../data/output/SA-2655.xlsx
 - ../data/output/SA-2660.xlsx
 - ../data/output/SA-2402.xlsx
 - ../data/ou